# Linear Regression in Practice: Evaluation and Overfitting

This notebook uses the public Ames Housing data to evaluate linear-regression models on data they did not see during training.

## Learning goals

- Load a public housing dataset with scikit-learn.
- Separate training and test data before fitting a model.
- Compare training and test error using RMSE and $R^2$.
- Identify overfitting when a flexible model performs well on training data but poorly on test data.
- Observe why a training set should cover the range of cases where predictions will be made.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

## Load public house-price data

The Ames Housing data records residential sales in Ames, Iowa. `GrLivArea` is the above-ground living area in square feet and `SalePrice` is the sale price in US dollars. This notebook uses only one feature, so its models are intentionally simplified.

Source: [OpenML house_prices dataset](https://www.openml.org/d/42165).

In [ ]:
ames = fetch_openml(data_id=42165, as_frame=True, parser='auto')
housing = ames.frame[['GrLivArea', 'SalePrice']].dropna().copy()
housing['GrLivArea_m2'] = housing['GrLivArea'].astype(float) * 0.092903
housing['SalePrice'] = housing['SalePrice'].astype(float)
housing = housing[(housing['GrLivArea_m2'] < 300) & (housing['SalePrice'] < 500_000)]

X = housing[['GrLivArea_m2']]
y = housing['SalePrice']
print(f'Usable observations: {len(housing)}')
housing.head()

## Training and test data

The test set must not influence model fitting or model selection. It represents future, unseen cases. Here, 25% of the observations are reserved as the test set before either model is trained.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print(f'Training observations: {len(X_train)}')
print(f'Test observations: {len(X_test)}')

## A reusable evaluation function

RMSE is the typical prediction error in dollars; lower is better. $R^2$ measures the proportion of variation explained by the model; higher is better.

In [ ]:
def report_scores(name, model, X_train, y_train, X_test, y_test):
    train_predictions = model.predict(X_train)
    test_predictions = model.predict(X_test)
    train_rmse = mean_squared_error(y_train, train_predictions) ** 0.5
    test_rmse = mean_squared_error(y_test, test_predictions) ** 0.5
    test_r2 = r2_score(y_test, test_predictions)
    print(f'{name}:')
    print(f'  training RMSE: ${train_rmse:,.0f}')
    print(f'  test RMSE:     ${test_rmse:,.0f}')
    print(f'  test R²:       {test_r2:.3f}')
    return train_rmse, test_rmse

## Compare a simple and a flexible model

A degree-1 model is a straight line. A degree-10 polynomial has many more terms, so it can bend to follow the training examples more closely. More flexibility is not automatically better.

In [ ]:
linear_model = LinearRegression().fit(X_train, y_train)
polynomial_model = make_pipeline(
    PolynomialFeatures(degree=10, include_bias=False),
    LinearRegression()
).fit(X_train, y_train)

linear_scores = report_scores('Linear model', linear_model, X_train, y_train, X_test, y_test)
polynomial_scores = report_scores('Degree-10 polynomial', polynomial_model, X_train, y_train, X_test, y_test)

In [ ]:
x_line = np.linspace(X['GrLivArea_m2'].min(), X['GrLivArea_m2'].max(), 300)
X_line = x_line.reshape(-1, 1)

plt.figure(figsize=(9, 5))
plt.scatter(X_train, y_train, s=16, alpha=0.45, label='Training data')
plt.scatter(X_test, y_test, s=16, alpha=0.45, label='Test data')
plt.plot(x_line, linear_model.predict(X_line), linewidth=3, label='Linear model')
plt.plot(x_line, polynomial_model.predict(X_line), linewidth=3, label='Degree-10 polynomial')
plt.xlabel('Above-ground living area (m²)')
plt.ylabel('Sale price (USD)')
plt.title('Training fit versus test-set performance')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## Interpreting overfitting

Overfitting occurs when a model learns noise or incidental details of the training data instead of the broader relationship. A warning sign is a low training RMSE paired with a noticeably higher test RMSE. Use the test result, rather than the training result alone, to decide which model generalizes better.

Try changing the polynomial degree from 10 to 2, 5, or 15. Which degree gives the best test RMSE?

## Why training data must cover the prediction range

The next comparison fits a line only to smaller homes and evaluates it on larger homes. This is not a random train/test split: it deliberately creates a distribution shift. The model must extrapolate beyond the sizes it observed during training.

In [ ]:
small_homes = housing[housing['GrLivArea_m2'] <= 120]
large_homes = housing[housing['GrLivArea_m2'] > 120]

range_model = LinearRegression().fit(small_homes[['GrLivArea_m2']], small_homes['SalePrice'])
report_scores(
    'Trained on homes up to 120 m²; tested on larger homes',
    range_model,
    small_homes[['GrLivArea_m2']], small_homes['SalePrice'],
    large_homes[['GrLivArea_m2']], large_homes['SalePrice']
)

## Summary

A training score tells us how well a model remembers its training examples. A test score estimates how well it will generalize to unseen data from the same setting. A good evaluation keeps the test data separate and checks that training data covers the cases where the model will be used.